## 0 · Setup — clone repo, install deps

**Environment:** you're driving a **Colab GPU runtime from VS Code**. Code runs on the Colab VM; files land under `/content/HE-IFD` — view/download results from the **VS Code remote Explorer** (right-click ▸ Download). Make sure the runtime is **GPU** (T4 is fine). The repo is public (no token).

In [ ]:
import os
if not os.path.isdir("/content/HE-IFD"):
    !git clone -q https://github.com/hkanpak21/HE-IFD.git /content/HE-IFD
%cd /content/HE-IFD
!git pull -q origin master
# torch/torchvision are preinstalled on Colab; add the rest:
!pip -q install transformers datasets timm
!git log --oneline -1

In [ ]:
import torch
ok = torch.cuda.is_available()
print("CUDA:", ok, "|", torch.cuda.get_device_name(0) if ok else "NO GPU — Runtime ▸ Change runtime type ▸ T4 GPU")

# 026 · Task-arithmetic λ scaling — cheap EVAL-ONLY verify

θ⋆(λ) = θ₀ + λ·Σ wᵢ·Δᵢ = (1−λ)·θ₀ + λ·θ⋆(1): pure interpolation between the basin (λ=0) and the current aggregate (λ=1) — one trajectory per cell, then 9 cheap evals. Mirrors `jobs/heifd_026_lambda_verify.sh`.

**Gate:** report the acc-vs-λ curve, λ⋆, and lift over λ=1. Do **not** launch a full λ grid from this — that's a separate decision.

In [ ]:
# Output roots. If you mounted Drive above it already set CACHE_ROOT/RESULTS_ROOT;
# otherwise these local (Colab VM) defaults apply.
import os
DATA_ROOT = "data"
CACHE_ROOT = globals().get("CACHE_ROOT", "cache")
RESULTS_ROOT = globals().get("RESULTS_ROOT", "results")
print("DATA_ROOT=%s  CACHE_ROOT=%s  RESULTS_ROOT=%s" % (DATA_ROOT, CACHE_ROOT, RESULTS_ROOT))

In [ ]:
import torchvision as tv
print("downloading datasets into", DATA_ROOT, "...")
tv.datasets.MNIST(DATA_ROOT, train=True, download=True)
tv.datasets.MNIST(DATA_ROOT, train=False, download=True)
tv.datasets.CIFAR100(DATA_ROOT, train=True, download=True)
tv.datasets.CIFAR100(DATA_ROOT, train=False, download=True)
print("vision datasets ready")

### Run — 4 cells (mlp_mnist, vit_b32_cifar100) × α{0.05,1.0}, λ∈{0…2}

In [ ]:
LAMBDAS = "0,0.25,0.5,0.75,1.0,1.25,1.5,1.75,2.0"
!python -m src.lambda_verify \
    --backbones mlp_mnist,vit_b32_cifar100 \
    --Ns 10 --alphas 0.05,1.0 --methods raw_union_K20 --seeds 42 --K 300 \
    --lambda-scales $LAMBDAS \
    --case heifd_026_lambda_verify \
    --data-root $DATA_ROOT --cache-root $CACHE_ROOT --results-root $RESULTS_ROOT

### Results — acc-vs-λ table + λ⋆ + lift

In [ ]:
from pathlib import Path
rm = Path(RESULTS_ROOT) / "heifd_026_lambda_verify" / "README.md"
print(rm.read_text() if rm.exists() else "(no README yet)")

In [ ]:
import json, glob
for p in sorted(glob.glob(f"{RESULTS_ROOT}/heifd_026_lambda_verify/cell_*.json")):
    d = json.load(open(p)); cur = d.get("lambda_curve") or []
    if not cur: continue
    best = max(cur, key=lambda x: x["acc"])
    a1 = next((c["acc"] for c in cur if abs(c["lambda"]-1.0)<1e-9), None)
    a0 = next((c["acc"] for c in cur if abs(c["lambda"]-0.0)<1e-9), None)
    print(f'{d["backbone"]:18s} a={d["alpha"]}  theta0(λ=0)={a0:.4f}  acc(λ=1)={a1:.4f}  λ*={best["lambda"]:.2f} acc(λ*)={best["acc"]:.4f}  lift={best["acc"]-a1:+.4f}')

In [ ]:
# Bundle results/heifd_026_lambda_verify/ for retrieval. In VS Code you can instead just
# right-click results/heifd_026_lambda_verify/ in the remote Explorer and Download.
import shutil
out = shutil.make_archive("/content/heifd_026_lambda_verify_results", "zip", f"{RESULTS_ROOT}/heifd_026_lambda_verify")
print("zipped ->", out, "\nDownload via the VS Code Explorer (right-click ▸ Download).")